# 04 -- LSEG IBES Data Collection

## Source
WRDS IBES (Institutional Brokers' Estimate System) via the `wrds` Python library, authenticated with username `henrylavender`. Data pulled from 2003-01-01 to 2024-12-31.

## Architecture
Three independent stock x month panels, each saved as a standalone parquet keyed on `(ticker, date)`, plus a linking table to map IBES tickers to CRSP PERMNOs. Total ~39 factors across three files.

## Design Choices

**Unadjusted tables:** The code uses unadjusted IBES tables (`statsum_xepsus`, `act_xepsus`, `statsumu_epsus`) rather than adjusted tables. Adjusted tables retroactively restate historical values for stock splits, which introduces look-ahead bias. Unadjusted preserves exactly what was known at the time.

**Point-in-time safety:** All `date` values are month-end timestamps representing when the information was known. For revenue, the anchor is `statpers` (the date IBES computed the consensus snapshot, typically the third Thursday of each month). For recommendations and price targets, it is the calendar month-end of the snapshot. Revenue surprises are explicitly gated so that `date >= anndats` before a surprise value is allowed, preventing look-ahead from unreported actuals.

**Staleness window:** Recommendations and price targets use a 12-month staleness window -- analyst opinions older than 12 months are dropped on the assumption the analyst is no longer actively covering the stock.

**FPI (Forecast Period Indicator):** Revenue estimates are pulled for FPI 1 (current fiscal year, primary signal), FPI 2 (next fiscal year, longer horizon), and FPI 6 (current fiscal quarter, most timely).

---

## Part 1: Revenue Estimates

### Tables Queried
- `ibes.statsum_xepsus` -- unadjusted summary statistics (consensus snapshots) filtered to `measure = 'SAL'` and `fpi IN ('1', '2', '6')`
- `ibes.act_xepsus` -- unadjusted actuals (reported revenue + announcement dates) filtered to `measure = 'SAL'`
- `ibes.statsumu_epsus` -- EPS summary statistics filtered to `measure = 'EPS'` and `fpi = '1'` (used solely for the revenue-EPS divergence calculation)

Note: Revenue (`SAL`) lives in the `_xepsus` tables, not the `_epsus` tables. The `_epsus` tables contain only `measure = 'EPS'`.

### Variables from API
- From `statsum_xepsus`: `ticker`, `fpedats`, `statpers`, `measure`, `fpi`, `numest`, `medest`, `meanest`, `stdev`, `highest`, `lowest`
- From `act_xepsus`: `ticker`, `pends`, `pdicity`, `value`, `anndats`
- From `statsumu_epsus`: `ticker`, `fpedats`, `statpers`, `fpi`, `meanest` (as `eps_meanest`)

### Derived Factors (Current Fiscal Year, FPI=1)
- `rev_mean_fy1` -- consensus mean revenue estimate
- `rev_numest` -- number of analysts covering revenue
- `rev_dispersion` -- analyst disagreement (stdev / |mean|)
- `rev_range` -- (highest - lowest) / |mean|
- `rev_revision_1m` -- 1-month % change in consensus mean (grouped by ticker and fpedats)
- `rev_revision_3m` -- 3-month % change in consensus mean
- `rev_numest_chg` -- month-over-month change in analyst count

### Derived Factors (Next Fiscal Year, FPI=2)
- `rev_fy2_revision_1m` -- 1-month % change in next-FY consensus mean
- `rev_fy2_dispersion` -- next-FY analyst disagreement

### Derived Factors (Current Quarter, FPI=6)
- `rev_q_revision_1m` -- 1-month % change in current-quarter consensus mean
- `rev_q_dispersion` -- current-quarter analyst disagreement

### Revenue Surprise
- Annual actuals (`pdicity = 'ANN'`) are merged onto the last consensus estimate before announcement
- `rev_surprise` -- actual minus last consensus
- `rev_surprise_pct` -- surprise as % of |last consensus|
- Look-ahead prevention: surprise values are set to NaN for any row where `date < anndats`

### Revenue-EPS Divergence
- `eps_revision_1m` is computed from the EPS summary table (1-month % change in EPS consensus mean, FPI=1)
- `rev_eps_divergence` -- revenue revision minus EPS revision; positive means revenue revision exceeds EPS revision, signaling organic top-line growth rather than cost-cutting

### Assembly
FPI=1 is the backbone. FPI=2, FPI=6, EPS revision, and surprise factors are left-joined on `(ticker, date)`. Deduplicated to one row per `(ticker, date)`, keeping the last row when multiple fiscal period end dates map to the same statpers.

---

## Part 2: Analyst Recommendations

### Table Queried
- `ibes.recddet` -- recommendation detail (individual analyst recommendations) filtered to `anndats BETWEEN '2003-01-01' AND '2024-12-31'`
- Columns: `ticker`, `estimid`, `amaskcd`, `ireccd`, `actdats`, `anndats`

### IBES Recommendation Codes
1 = Strong Buy, 2 = Buy, 3 = Hold, 4 = Underperform, 5 = Sell (lower = more bullish)

### Monthly Snapshot Construction
For each month-end date from 2003-01 to 2024-12:
1. Find all recommendations announced on or before that month-end
2. Keep only the most recent recommendation per analyst per ticker (each rec is active from `anndats` until the same analyst issues a new rec for that ticker)
3. Drop recommendations older than 12 months (staleness filter)
4. Compute cross-sectional statistics across the active set for each ticker

### Derived Factors (Snapshot-Based)
- `rec_mean` -- mean recommendation code across active analysts
- `rec_median` -- median recommendation code
- `rec_numrec` -- count of active recommendations
- `rec_buy_pct` -- % of analysts with Buy or Strong Buy (ireccd <= 2)
- `rec_sell_pct` -- % of analysts with Sell or Underperform (ireccd >= 4)
- `rec_buy_sell_spread` -- buy % minus sell %
- `rec_dispersion` -- standard deviation of recommendation codes
- `rec_revision` -- 1-month change in `rec_mean` (within ticker)
- `rec_revision_3m` -- 3-month change in `rec_mean`

### Derived Factors (Change-Based)
Each recommendation is compared to the same analyst's prior recommendation for the same ticker:
- `rec_upgrades` -- count of upgrades in the calendar month (new rec < prior rec)
- `rec_downgrades` -- count of downgrades in the calendar month
- `rec_changes` -- total recommendation changes in the month
- `rec_breadth` -- (upgrades - downgrades) / total changes

### Note on `rec_up_down_ratio`
Originally computed as upgrades / downgrades, but dropped in the post-collection cleanup because NaN cases are ambiguous: 0/0 (neutral) and N/0 (extremely bullish) cannot be distinguished after the fact. `rec_breadth` handles this correctly since it is bounded [-1, +1].

---

## Part 3: Price Targets

### Table Queried
- `ibes.ptgdet` -- price target detail (individual analyst targets) filtered to `anndats BETWEEN '2003-01-01' AND '2024-12-31'` and `curr = 'USD'`
- Columns: `ticker`, `estimid`, `amaskcd`, `value`, `anndats`, `horizon`, `curr`

### Monthly Snapshot Construction
Same logic as recommendations: for each month-end, keep the most recent price target per analyst per ticker, drop targets older than 12 months and targets with value <= 0.

### Derived Factors
- `ptg_mean` -- consensus mean price target ($)
- `ptg_median` -- median price target ($)
- `ptg_high` -- highest analyst target ($)
- `ptg_low` -- lowest analyst target ($)
- `ptg_numest` -- number of active price targets
- `ptg_dispersion` -- std / mean
- `ptg_range` -- (high - low) / mean
- `ptg_upside_skew` -- (high - median) / (median - low); >1 means analysts see more upside than downside
- `ptg_revision` -- 1-month % change in consensus mean target
- `ptg_revision_3m` -- 3-month % change in consensus mean target
- `ptg_numest_chg` -- change in analyst count

### Note on Implied Return
The most powerful price target signal -- `implied_return = (ptg_mean / current_price) - 1` -- is deliberately not computed here because it requires stock prices, which would introduce a CRSP merge dependency. It is computed downstream in the pipeline.

---

## Part 4: IBES Ticker to PERMNO Linking Table

### Tables Queried
- `ibes.idsum` -- IBES identifier summary (maps IBES ticker to CUSIP), filtered to `usfirm = 1`
- `crsp.stocknames` -- CRSP stock names (maps NCUSIP to PERMNO)
- Joined on `ibes.idsum.cusip = crsp.stocknames.ncusip`

### Columns
- `ticker` -- IBES ticker
- `ibes_cusip` -- CUSIP from IBES
- `permno` -- CRSP PERMNO
- `ncusip` -- CRSP NCUSIP
- `sdates` -- link start date
- `edates` -- link end date (derived as the next row's `sdates` within the same ticker, or 2099-12-31 if no subsequent row)

---

## Post-Collection Cleanup (Separate Script)

### Revenue
- `rev_surprise` and `rev_surprise_pct` are forward-filled within each ticker with `limit=4` (approximately one earnings cycle). This matches how OAP computes `RevenueSurprise` -- the most recent surprise carries forward until a new actual arrives. The limit of 4 prevents feeding stale data from delisted or acquired companies.

### Recommendations
- `rec_up_down_ratio` is dropped entirely (ambiguous NaN semantics as described above)
- `rec_breadth` NaN values are filled with 0 (structural zero: no recommendation changes in the month means neutral signal, not missing data)

---

## Relationship to OpenAssetPricing (OAP) Factors
A comparison was conducted between IBES-collected factors and existing OAP factors. The result:
- **IBES replaces 4 OAP factors** with dramatically better coverage: `ConsRecomm` (86.5% NaN in OAP vs 0.0% in IBES `rec_mean`), `ChangeInRecommendation` (39.6% vs 1.2%), `UpRecomm` (39.6% vs 0.0%), `DownRecomm` (39.6% vs 0.0%)
- **OAP retained for**: `RevenueSurprise` (12.1% NaN vs 99% in raw IBES), all 10 EPS-based factors (`FEPS`, `ForecastDispersion`, `AnalystRevision`, `REV6`, `EarningsSurprise`, `fgr5yrLag`, `AOP`, `AnalystValue`, `EarningsForecastDisparity`, `PredictedFE`)
- **~29 new factors from IBES** with no OAP equivalent, particularly revenue revision/dispersion signals and all price target factors

## Outputs
- `Data/Data_Collection/Initial/04_LSEG_IBES/ibes_revenue.parquet` -- 14 revenue factors, keyed on (ticker, date)
- `Data/Data_Collection/Initial/04_LSEG_IBES/ibes_recommendations.parquet` -- 13 recommendation factors, keyed on (ticker, date)
- `Data/Data_Collection/Initial/04_LSEG_IBES/ibes_price_targets.parquet` -- 11 price target factors, keyed on (ticker, date)
- `Data/Data_Collection/Initial/04_LSEG_IBES/ibes_permno_link.parquet` -- IBES ticker to CRSP PERMNO crosswalk

In [ ]:
"""
LSEG IBES Data Collection — Revenue Estimates, Recommendations, Price Targets
==============================================================================
Pulls three orthogonal data sources from WRDS IBES, computes stock-month
factor panels, and saves as standalone parquet files.

Outputs:
  data/ibes_revenue.parquet      — Revenue estimate factors (stock × month)
  data/ibes_recommendations.parquet — Analyst recommendation factors (stock × month)
  data/ibes_price_targets.parquet   — Price target factors (stock × month)

Design Choices & Notes:
───────────────────────
1. UNADJUSTED tables ('statsumu_epsus', 'actu_epsus') are used for revenue.
   The adjusted tables retroactively restate historical values for splits,
   which introduces look-ahead bias. Unadjusted preserves point-in-time data.

2. For revenue summary stats, 'statpers' is the IBES "statistical period" —
   the date on which IBES computed the consensus snapshot, typically the
   third Thursday of each month. This is our point-in-time anchor.
   'fpedats' is the fiscal period end date being forecasted.

3. FPI (Forecast Period Indicator):
     1 = Current fiscal YEAR      (most coverage, primary signal)
     2 = Next fiscal YEAR         (longer horizon, less coverage)
     6 = Current fiscal QUARTER   (most timely, shorter horizon)
   We pull FPI 1 and 2 for annual, FPI 6 for quarterly.

4. We save IBES ticker as the identifier. Will need to link to the
   equity universe via ibes.idsum → CUSIP → CRSP permno. A linking
   helper is included at the end of this script.

5. All dates are ORIGINAL from IBES, untouched. The 'statpers' (for revenue)
   and the month-end grouping dates are point-in-time safe — they represent
   when the information was known, not when the fiscal period ends.

6. Revenue actuals use 'anndats' (announcement date) to ensure we only
   compute surprises AFTER the actual was publicly released.
"""

import wrds
import pandas as pd
import numpy as np
from pathlib import Path
import time



db = wrds.Connection(wrds_username='henrylavender')

START = '2003-01-01'
END   = '2024-12-31'

# ═══════════════════════════════════════════════════════════════════════════════
# HELPER: Inspect table schema
# ═══════════════════════════════════════════════════════════════════════════════
def show_columns(db, library, table, n_sample=3):
    """Print columns and sample values for a WRDS table."""
    cols = db.describe_table(library, table)
    print(f"\n  {library}.{table} — {len(cols)} columns:")
    for _, row in cols.iterrows():
        print(f"    {row['name']:<40s} {str(row['type']):<15s}")
    # Sample
    sample = db.raw_sql(f"SELECT * FROM {library}.{table} LIMIT {n_sample}")
    print(f"\n  Sample ({n_sample} rows):")
    print(sample.to_string(index=False))
    return cols


# ═══════════════════════════════════════════════════════════════════════════════
# PART 1: REVENUE ESTIMATES
# ═══════════════════════════════════════════════════════════════════════════════
#
# Why revenue is orthogonal to existing EPS factors:
#   - EPS can be managed via buybacks, cost-cutting, one-time items, accounting.
#   - Revenue (top-line) is much harder to manipulate.
#   - Revenue revisions that diverge from EPS revisions are powerful quality signals.
#   - Academic literature (Jegadeesh & Livnat 2006, Ertimur et al. 2003) shows
#     revenue surprises predict returns incremental to earnings surprises.
#
# Source tables:
#   ibes.statsumu_epsus — Unadjusted summary statistics (consensus snapshots)
#   ibes.actu_epsus     — Unadjusted actuals (reported numbers + announcement dates)
#
# ═══════════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("PART 1: REVENUE ESTIMATES")
print("=" * 80)

# ── 1a. Pull revenue consensus snapshots ─────────────────────────────────────
# 'measure' = 'SAL' selects sales/revenue (vs 'EPS' for earnings).
# We pull FPI 1 (current FY), 2 (next FY), 6 (current quarter).
# statpers = date IBES computed the consensus (point-in-time safe).
# fpedats  = fiscal period end date being forecasted.

print("\nPulling revenue summary statistics...")
# NOTE: Revenue (SAL) lives in ibes.statsumu_xepsus, NOT ibes.statsumu_epsus.
# The _epsus tables contain ONLY measure='EPS'. The _xepsus tables contain
# all OTHER measures: 'SAL' (Sales), 'CPS' (Cash Flow), 'EBD' (EBITDA), etc.
rev_summary = db.raw_sql(f"""
    SELECT ticker, fpedats, statpers, measure, fpi,
           numest, medest, meanest, stdev, highest, lowest
    FROM ibes.statsum_xepsus
    WHERE measure = 'SAL'
      AND fpi IN ('1', '2', '6')
      AND statpers BETWEEN '{START}' AND '{END}'
""")
print(f"  Revenue summary: {rev_summary.shape[0]:,} rows, "
      f"{rev_summary['ticker'].nunique():,} tickers")
print(f"  FPI distribution:\n{rev_summary['fpi'].value_counts().to_string()}")

# ── 1b. Pull revenue actuals ────────────────────────────────────────────────
# anndats = date the actual revenue was publicly announced.
# We need this for surprise calculations AND to prevent look-ahead bias
# (only use actuals after anndats).

print("\nPulling revenue actuals...")
# Same logic: revenue actuals are in ibes.actu_xepsus, not ibes.actu_epsus.
rev_actuals = db.raw_sql(f"""
    SELECT ticker, pends, pdicity, value, anndats
    FROM ibes.act_xepsus
    WHERE measure = 'SAL'
      AND pends BETWEEN '{START}' AND '{END}'
""")
print(f"  Revenue actuals: {rev_actuals.shape[0]:,} rows")

# ── 1c. Also pull EPS summary for revenue-EPS divergence factors ────────────
# We only need mean and revision for the divergence calculation.
# If you already have this data elsewhere, you can skip this query.

print("\nPulling EPS summary (for revenue-EPS divergence)...")
eps_summary = db.raw_sql(f"""
    SELECT ticker, fpedats, statpers, fpi, meanest AS eps_meanest
    FROM ibes.statsumu_epsus
    WHERE measure = 'EPS'
      AND fpi = '1'
      AND statpers BETWEEN '{START}' AND '{END}'
""")
print(f"  EPS summary: {eps_summary.shape[0]:,} rows")

# ── 1d. Compute revenue factors ─────────────────────────────────────────────
# Strategy: Group by (ticker, statpers, fpi) to get one row per stock-month-horizon.
# Then compute cross-horizon and time-series features.

print("\nComputing revenue factors...")

# Convert types
for df in [rev_summary, rev_actuals, eps_summary]:
    for col in df.columns:
        if col in ['fpedats', 'statpers', 'pends', 'anndats']:
            df[col] = pd.to_datetime(df[col])

for col in ['numest', 'medest', 'meanest', 'stdev', 'highest', 'lowest']:
    rev_summary[col] = pd.to_numeric(rev_summary[col], errors='coerce')

# ── Current fiscal year (FPI=1) — primary revenue factors ────────────────────
rev_fy1 = rev_summary[rev_summary['fpi'] == '1'].copy()
rev_fy1 = rev_fy1.sort_values(['ticker', 'fpedats', 'statpers'])

# Revenue Dispersion: stdev / |mean|
# High dispersion = high analyst disagreement about top-line growth.
# Empirically a negative return predictor (Diether et al. 2002 logic extends to revenue).
rev_fy1['rev_dispersion'] = rev_fy1['stdev'] / rev_fy1['meanest'].abs().replace(0, np.nan)

# Revenue Range: (highest - lowest) / |mean|
# Alternative uncertainty measure, less sensitive to outliers than stdev.
rev_fy1['rev_range'] = (
    (rev_fy1['highest'] - rev_fy1['lowest']) /
    rev_fy1['meanest'].abs().replace(0, np.nan)
)

# Revenue Revision (1-month): % change in consensus mean vs prior month.
# This is the revenue analogue of your AnalystRevision factor.
# Positive revision = analysts raising top-line estimates = bullish signal.
rev_fy1['rev_revision_1m'] = (
    rev_fy1.groupby(['ticker', 'fpedats'])['meanest']
    .pct_change() * 100
)

# Revenue Revision (3-month): % change over 3 months for slower-moving signal.
rev_fy1['rev_revision_3m'] = (
    rev_fy1.groupby(['ticker', 'fpedats'])['meanest']
    .pct_change(periods=3) * 100
)

# Number of analysts covering revenue (breadth).
# More analysts = more liquid, better covered name.
# Changes in coverage can signal increasing/decreasing institutional interest.
rev_fy1['rev_numest'] = rev_fy1['numest']

# Coverage change: month-over-month change in number of analysts.
rev_fy1['rev_numest_chg'] = (
    rev_fy1.groupby(['ticker', 'fpedats'])['numest'].diff()
)

# ── Next fiscal year (FPI=2) — longer-horizon factors ────────────────────────
rev_fy2 = rev_summary[rev_summary['fpi'] == '2'].copy()
rev_fy2 = rev_fy2.sort_values(['ticker', 'fpedats', 'statpers'])

rev_fy2['rev_fy2_revision_1m'] = (
    rev_fy2.groupby(['ticker', 'fpedats'])['meanest']
    .pct_change() * 100
)
rev_fy2['rev_fy2_dispersion'] = (
    rev_fy2['stdev'] / rev_fy2['meanest'].abs().replace(0, np.nan)
)

# Rename for merge
rev_fy2_slim = rev_fy2[['ticker', 'statpers', 'rev_fy2_revision_1m',
                         'rev_fy2_dispersion']].copy()
rev_fy2_slim = rev_fy2_slim.rename(columns={'statpers': 'statpers'})

# ── Current quarter (FPI=6) — most timely signal ────────────────────────────
rev_q = rev_summary[rev_summary['fpi'] == '6'].copy()
rev_q = rev_q.sort_values(['ticker', 'fpedats', 'statpers'])

rev_q['rev_q_revision_1m'] = (
    rev_q.groupby(['ticker', 'fpedats'])['meanest']
    .pct_change() * 100
)
rev_q['rev_q_dispersion'] = (
    rev_q['stdev'] / rev_q['meanest'].abs().replace(0, np.nan)
)

rev_q_slim = rev_q[['ticker', 'statpers', 'rev_q_revision_1m',
                      'rev_q_dispersion']].copy()

# ── Revenue Surprise ─────────────────────────────────────────────────────────
# Merge actuals onto the LAST consensus before the announcement.
# Only count a surprise AFTER the announcement date (anndats).
# This factor is backward-looking: it tells you the most recent revenue surprise,
# which has momentum implications (post-revenue-announcement drift).

# For annual actuals
rev_act_ann = rev_actuals[rev_actuals['pdicity'] == 'ANN'].copy()
rev_act_ann = rev_act_ann.rename(columns={'pends': 'fpedats', 'value': 'rev_actual'})
rev_act_ann = rev_act_ann[['ticker', 'fpedats', 'rev_actual', 'anndats']]
rev_act_ann = rev_act_ann.drop_duplicates(subset=['ticker', 'fpedats'], keep='last')

# Get the last consensus before announcement
rev_last_est = (
    rev_fy1[rev_fy1['meanest'].notna()]
    .sort_values('statpers')
    .drop_duplicates(subset=['ticker', 'fpedats'], keep='last')
    [['ticker', 'fpedats', 'meanest']]
    .rename(columns={'meanest': 'rev_last_consensus'})
)

rev_surprise = rev_act_ann.merge(rev_last_est, on=['ticker', 'fpedats'], how='inner')
rev_surprise['rev_surprise'] = rev_surprise['rev_actual'] - rev_surprise['rev_last_consensus']
rev_surprise['rev_surprise_pct'] = (
    rev_surprise['rev_surprise'] /
    rev_surprise['rev_last_consensus'].abs().replace(0, np.nan) * 100
)

# ── Revenue-EPS Revision Divergence ──────────────────────────────────────────
# When revenue revision is negative but EPS revision is positive (or flat),
# it signals cost-cutting rather than organic growth — a quality red flag.
# Conversely, revenue up + EPS down might signal investment for future growth.

eps_summary['eps_meanest'] = pd.to_numeric(eps_summary['eps_meanest'], errors='coerce')
eps_rev = eps_summary.sort_values(['ticker', 'fpedats', 'statpers'])
eps_rev['eps_revision_1m'] = (
    eps_rev.groupby(['ticker', 'fpedats'])['eps_meanest']
    .pct_change() * 100
)
eps_rev_slim = eps_rev[['ticker', 'statpers', 'eps_revision_1m']].copy()

# ── Assemble final revenue factor panel ──────────────────────────────────────
# Use FPI=1 as the backbone, merge in FPI=2, FPI=6, and divergence factors.

rev_factors = rev_fy1[[
    'ticker', 'statpers', 'fpedats', 'meanest', 'numest',
    'rev_dispersion', 'rev_range', 'rev_revision_1m', 'rev_revision_3m',
    'rev_numest', 'rev_numest_chg'
]].copy()
rev_factors = rev_factors.rename(columns={
    'meanest': 'rev_mean_fy1',
    'statpers': 'date',
})

# Merge FPI=2 factors
rev_factors = rev_factors.merge(
    rev_fy2_slim, left_on=['ticker', 'date'], right_on=['ticker', 'statpers'],
    how='left'
).drop(columns=['statpers'], errors='ignore')

# Merge quarterly factors
rev_factors = rev_factors.merge(
    rev_q_slim, left_on=['ticker', 'date'], right_on=['ticker', 'statpers'],
    how='left'
).drop(columns=['statpers'], errors='ignore')

# Merge EPS revision for divergence
rev_factors = rev_factors.merge(
    eps_rev_slim, left_on=['ticker', 'date'], right_on=['ticker', 'statpers'],
    how='left'
).drop(columns=['statpers'], errors='ignore')

# Revenue-EPS divergence: positive = revenue revision exceeds EPS revision
rev_factors['rev_eps_divergence'] = (
    rev_factors['rev_revision_1m'] - rev_factors['eps_revision_1m']
)

# Drop intermediate column
rev_factors = rev_factors.drop(columns=['eps_revision_1m'], errors='ignore')

# ── Merge revenue surprise (keyed on fpedats, lagged by announcement date) ──
# We assign the surprise to all months AFTER the announcement.
# This creates a "most recent surprise" feature that updates when new actuals arrive.
rev_surprise_slim = rev_surprise[
    ['ticker', 'fpedats', 'rev_surprise', 'rev_surprise_pct', 'anndats']
].copy()

rev_factors = rev_factors.merge(
    rev_surprise_slim, on=['ticker', 'fpedats'], how='left'
)
# Zero out surprises that haven't been announced yet (look-ahead prevention)
mask = rev_factors['anndats'].notna() & (rev_factors['date'] < rev_factors['anndats'])
rev_factors.loc[mask, ['rev_surprise', 'rev_surprise_pct']] = np.nan
rev_factors = rev_factors.drop(columns=['anndats', 'fpedats'], errors='ignore')

# ── Deduplicate: keep one row per (ticker, date) ────────────────────────────
# Multiple fpedats can map to the same statpers (e.g., when fiscal year rolls).
# Keep the row for the nearest fiscal period end.
rev_factors = rev_factors.sort_values(['ticker', 'date']).reset_index(drop=True)
rev_factors = rev_factors.drop_duplicates(subset=['ticker', 'date'], keep='last')

# ── Save ─────────────────────────────────────────────────────────────────────
rev_factors.to_parquet('../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_revenue.parquet', index=False, engine='pyarrow')
rev_cols = [c for c in rev_factors.columns if c not in ['ticker', 'date']]
print(f"\nSaved ../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_revenue.parquet: {rev_factors.shape}")
print(f"  Tickers: {rev_factors['ticker'].nunique():,}")
print(f"  Date range: {rev_factors['date'].min().date()} → {rev_factors['date'].max().date()}")
print(f"  Factors ({len(rev_cols)}):")
for c in rev_cols:
    pct_nan = rev_factors[c].isna().mean() * 100
    print(f"    {c:<30s} — {pct_nan:>5.1f}% NaN")


# ═══════════════════════════════════════════════════════════════════════════════
# PART 2: ANALYST RECOMMENDATIONS
# ═══════════════════════════════════════════════════════════════════════════════
#
# Why orthogonal to the EPS factors:
#   - EPS factors measure what analysts EXPECT to happen (quantitative forecast).
#   - Recommendations measure what analysts ADVISE clients to do (qualitative judgment).
#   - An analyst can maintain an EPS estimate while changing a recommendation,
#     or vice versa. The signals are empirically distinct.
#   - Recommendation changes (Womack 1996, Jegadeesh et al. 2004) predict returns
#     incremental to earnings revisions.
#
# Source: ibes.recddet (recommendation detail — individual analyst recs)
#   We aggregate ourselves rather than using a summary table to have full
#   control over the lookback window, handling of stale recs, etc.
#
# IBES recommendation codes (ireccd):
#   1 = Strong Buy, 2 = Buy, 3 = Hold, 4 = Underperform, 5 = Sell
#   Lower = more bullish. Mean ~2.0 = very bullish consensus.
#
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'=' * 80}")
print("PART 2: ANALYST RECOMMENDATIONS")
print("=" * 80)

# ── 2a. Pull recommendation detail data ─────────────────────────────────────
# anndats = date the recommendation was announced (point-in-time safe).
# actdats = date the recommendation became active.
# We use anndats for dating to prevent look-ahead bias.

print("\nPulling recommendation detail data...")
recs = db.raw_sql(f"""
    SELECT ticker, estimid, amaskcd, ireccd, actdats, anndats
    FROM ibes.recddet
    WHERE anndats BETWEEN '{START}' AND '{END}'
""")
recs['anndats'] = pd.to_datetime(recs['anndats'])
recs['actdats'] = pd.to_datetime(recs['actdats'])
recs['ireccd'] = pd.to_numeric(recs['ireccd'], errors='coerce')
print(f"  Recommendations: {recs.shape[0]:,} rows, "
      f"{recs['ticker'].nunique():,} tickers")
print(f"  Rec code distribution:\n{recs['ireccd'].value_counts().sort_index().to_string()}")

# ── 2b. Build monthly recommendation snapshots ──────────────────────────────
# For each stock-month, we look at all ACTIVE recommendations.
# A recommendation stays active until the same analyst issues a new one
# or it becomes stale (we use a 12-month staleness window).
#
# Approach:
#   1. For each (ticker, month_end), find all recs announced on or before month_end
#      whose announcing analyst hasn't issued a newer rec for that ticker.
#   2. Drop recs older than 12 months (stale).
#   3. Compute cross-sectional statistics on the active set.

print("\nBuilding monthly recommendation snapshots...")

# Create month-end dates for grouping
recs['month'] = recs['anndats'].dt.to_period('M').dt.to_timestamp('M')

# Get the most recent rec per (ticker, analyst) as of each month
# This is the expensive step — we expand to a ticker×month panel.

# Unique months and tickers
months = pd.date_range(START, END, freq='ME')
tickers = recs['ticker'].unique()

# Efficient approach: for each rec, it's "active" from anndats until replaced.
# Sort and keep last rec per analyst-ticker before each month.
recs_sorted = recs.sort_values(['ticker', 'estimid', 'amaskcd', 'anndats'])

# Mark the "end date" of each rec (when that analyst next updates this ticker)
recs_sorted['next_rec_date'] = (
    recs_sorted.groupby(['ticker', 'estimid', 'amaskcd'])['anndats']
    .shift(-1)
)
# If no next rec, stays active (we'll apply staleness filter)
recs_sorted['expires'] = recs_sorted['next_rec_date'].fillna(pd.Timestamp('2099-12-31'))

# For each month, find active recs
rec_snapshots = []

for m in months:
    # Active = announced on or before month_end, not yet replaced, not stale (>12mo)
    stale_cutoff = m - pd.DateOffset(months=12)
    active = recs_sorted[
        (recs_sorted['anndats'] <= m) &
        (recs_sorted['expires'] > m) &
        (recs_sorted['anndats'] > stale_cutoff)
    ].copy()

    if len(active) == 0:
        continue

    # Aggregate per ticker
    grp = active.groupby('ticker')['ireccd']
    snap = pd.DataFrame({
        'rec_mean':     grp.mean(),
        'rec_median':   grp.median(),
        'rec_std':      grp.std(),
        'rec_numrec':   grp.count(),
        'rec_buy_count':  active[active['ireccd'] <= 2].groupby('ticker')['ireccd'].count(),
        'rec_sell_count': active[active['ireccd'] >= 4].groupby('ticker')['ireccd'].count(),
        'rec_hold_count': active[active['ireccd'] == 3].groupby('ticker')['ireccd'].count(),
    })
    snap['date'] = m
    snap = snap.reset_index()
    rec_snapshots.append(snap)

    if m.month == 12:
        print(f"  Processed through {m.date()}, "
              f"{len(rec_snapshots)} months done")

rec_panel = pd.concat(rec_snapshots, ignore_index=True)
rec_panel = rec_panel.fillna({'rec_buy_count': 0, 'rec_sell_count': 0, 'rec_hold_count': 0})

# ── 2c. Compute recommendation factors ──────────────────────────────────────

# Buy percentage: fraction of analysts with buy/strong buy.
# High buy % = bullish consensus; extreme values may be contrarian signal.
rec_panel['rec_buy_pct'] = rec_panel['rec_buy_count'] / rec_panel['rec_numrec'] * 100

# Sell percentage: fraction with underperform/sell.
rec_panel['rec_sell_pct'] = rec_panel['rec_sell_count'] / rec_panel['rec_numrec'] * 100

# Buy-Sell spread: net bullishness of the analyst community.
rec_panel['rec_buy_sell_spread'] = rec_panel['rec_buy_pct'] - rec_panel['rec_sell_pct']

# Dispersion: std dev of recommendation codes. High = disagreement among analysts.
rec_panel['rec_dispersion'] = rec_panel['rec_std']

# ── Recommendation revision (change in consensus vs prior month) ─────────────
rec_panel = rec_panel.sort_values(['ticker', 'date'])
rec_panel['rec_revision'] = rec_panel.groupby('ticker')['rec_mean'].diff()

# 3-month revision
rec_panel['rec_revision_3m'] = rec_panel.groupby('ticker')['rec_mean'].diff(3)

# ── Count recent upgrades and downgrades (trailing 30 days) ──────────────────
# An upgrade = analyst's new rec is lower number (more bullish) than their previous.
# A downgrade = higher number (more bearish).

print("\nComputing upgrade/downgrade counts...")

# Compare each rec to the analyst's prior rec for that ticker
recs_changes = recs_sorted.copy()
recs_changes['prev_rec'] = (
    recs_changes.groupby(['ticker', 'estimid', 'amaskcd'])['ireccd']
    .shift(1)
)
recs_changes['is_upgrade']   = (recs_changes['ireccd'] < recs_changes['prev_rec']).astype(float)
recs_changes['is_downgrade'] = (recs_changes['ireccd'] > recs_changes['prev_rec']).astype(float)
recs_changes['is_change']    = (recs_changes['ireccd'] != recs_changes['prev_rec']).astype(float)

# Aggregate to stock-month: count upgrades/downgrades in that calendar month
recs_changes['month'] = recs_changes['anndats'].dt.to_period('M').dt.to_timestamp('M')
chg_monthly = recs_changes.groupby(['ticker', 'month']).agg(
    rec_upgrades   = ('is_upgrade', 'sum'),
    rec_downgrades = ('is_downgrade', 'sum'),
    rec_changes    = ('is_change', 'sum'),
).reset_index().rename(columns={'month': 'date'})

# Upgrade/downgrade ratio: > 1 means more upgrades than downgrades
chg_monthly['rec_up_down_ratio'] = (
    chg_monthly['rec_upgrades'] /
    chg_monthly['rec_downgrades'].replace(0, np.nan)
)

# Net upgrade breadth: (upgrades - downgrades) / total changes
chg_monthly['rec_breadth'] = (
    (chg_monthly['rec_upgrades'] - chg_monthly['rec_downgrades']) /
    chg_monthly['rec_changes'].replace(0, np.nan)
)

# Merge into main panel
rec_panel = rec_panel.merge(chg_monthly, on=['ticker', 'date'], how='left')
rec_panel = rec_panel.fillna({
    'rec_upgrades': 0, 'rec_downgrades': 0, 'rec_changes': 0
})

# ── Drop intermediate columns, save ─────────────────────────────────────────
rec_panel = rec_panel.drop(columns=[
    'rec_buy_count', 'rec_sell_count', 'rec_hold_count', 'rec_std'
], errors='ignore')

rec_panel = rec_panel.drop_duplicates(subset=['ticker', 'date'], keep='last')
rec_panel.to_parquet('../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_recommendations.parquet', index=False, engine='pyarrow')

rec_cols = [c for c in rec_panel.columns if c not in ['ticker', 'date']]
print(f"\nSaved ../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_recommendations.parquet: {rec_panel.shape}")
print(f"  Tickers: {rec_panel['ticker'].nunique():,}")
print(f"  Date range: {rec_panel['date'].min().date()} → {rec_panel['date'].max().date()}")
print(f"  Factors ({len(rec_cols)}):")
for c in rec_cols:
    pct_nan = rec_panel[c].isna().mean() * 100
    print(f"    {c:<30s} — {pct_nan:>5.1f}% NaN")


# ═══════════════════════════════════════════════════════════════════════════════
# PART 3: PRICE TARGETS
# ═══════════════════════════════════════════════════════════════════════════════
#
# Why orthogonal:
#   - EPS factors = what analysts expect the company will EARN.
#   - Recommendations = qualitative BUY/SELL advice.
#   - Price targets = what analysts think the STOCK is WORTH.
#   - A price target embeds the analyst's view on both fundamentals AND valuation.
#   - Brav & Lehavy (2003) show price targets predict returns incrementally.
#   - The implied upside (target / current price) is a direct valuation signal.
#
# Source: ibes.ptgdet (price target detail — individual analyst targets)
#   We aggregate to monthly consensus ourselves.
#
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'=' * 80}")
print("PART 3: PRICE TARGETS")
print("=" * 80)

# ── 3a. Pull price target detail ────────────────────────────────────────────
# value  = the analyst's price target (in local currency).
# anndats = announcement date (point-in-time safe).
# horizon = target horizon in months (typically 12).
# curr   = currency code.

print("\nPulling price target detail data...")
# Note: ibes.ptgdet has no 'expdt' column. Available columns are:
# ticker, cusip, oftic, cname, actdats, estimid, alysnam, horizon,
# value, estcur, curr, amaskcd, usfirm, measure, acttims, anndats, anntims
ptg = db.raw_sql(f"""
    SELECT ticker, estimid, amaskcd, value, anndats, horizon, curr
    FROM ibes.ptgdet
    WHERE anndats BETWEEN '{START}' AND '{END}'
      AND curr = 'USD'
""")
ptg['anndats'] = pd.to_datetime(ptg['anndats'])
ptg['value']   = pd.to_numeric(ptg['value'], errors='coerce')
print(f"  Price targets: {ptg.shape[0]:,} rows, "
      f"{ptg['ticker'].nunique():,} tickers")

# ── 3b. Build monthly price target snapshots ────────────────────────────────
# Same logic as recommendations: keep most recent target per analyst,
# drop stale targets (>12 months old).

print("\nBuilding monthly price target snapshots...")

ptg_sorted = ptg.sort_values(['ticker', 'estimid', 'amaskcd', 'anndats'])

# Mark expiry (when analyst issues next target for this ticker)
ptg_sorted['next_ptg_date'] = (
    ptg_sorted.groupby(['ticker', 'estimid', 'amaskcd'])['anndats']
    .shift(-1)
)
ptg_sorted['expires'] = ptg_sorted['next_ptg_date'].fillna(pd.Timestamp('2099-12-31'))

ptg_snapshots = []

for m in months:
    stale_cutoff = m - pd.DateOffset(months=12)
    active = ptg_sorted[
        (ptg_sorted['anndats'] <= m) &
        (ptg_sorted['expires'] > m) &
        (ptg_sorted['anndats'] > stale_cutoff) &
        (ptg_sorted['value'] > 0)  # drop zero/negative targets
    ].copy()

    if len(active) == 0:
        continue

    grp = active.groupby('ticker')['value']
    snap = pd.DataFrame({
        'ptg_mean':   grp.mean(),
        'ptg_median': grp.median(),
        'ptg_high':   grp.max(),
        'ptg_low':    grp.min(),
        'ptg_std':    grp.std(),
        'ptg_numest': grp.count(),
    })
    snap['date'] = m
    snap = snap.reset_index()
    ptg_snapshots.append(snap)

    if m.month == 12:
        print(f"  Processed through {m.date()}, "
              f"{len(ptg_snapshots)} months done")

ptg_panel = pd.concat(ptg_snapshots, ignore_index=True)

# ── 3c. Compute price target factors ────────────────────────────────────────

# Dispersion: std / mean. High = analysts disagree on fair value.
ptg_panel['ptg_dispersion'] = (
    ptg_panel['ptg_std'] / ptg_panel['ptg_mean'].replace(0, np.nan)
)

# Range: (high - low) / mean. Normalized spread of targets.
ptg_panel['ptg_range'] = (
    (ptg_panel['ptg_high'] - ptg_panel['ptg_low']) /
    ptg_panel['ptg_mean'].replace(0, np.nan)
)

# Upside skew: (high - median) / (median - low).
# > 1 means analysts see more upside than downside.
ptg_panel['ptg_upside_skew'] = (
    (ptg_panel['ptg_high'] - ptg_panel['ptg_median']) /
    (ptg_panel['ptg_median'] - ptg_panel['ptg_low']).replace(0, np.nan)
)

# Consensus target revision: month-over-month change in mean target.
ptg_panel = ptg_panel.sort_values(['ticker', 'date'])
ptg_panel['ptg_revision'] = ptg_panel.groupby('ticker')['ptg_mean'].pct_change() * 100
ptg_panel['ptg_revision_3m'] = ptg_panel.groupby('ticker')['ptg_mean'].pct_change(3) * 100

# Coverage change
ptg_panel['ptg_numest_chg'] = ptg_panel.groupby('ticker')['ptg_numest'].diff()

# ── NOTE ON IMPLIED RETURN ───────────────────────────────────────────────────
# The most powerful price target factor is (consensus target / current price) - 1,
# which gives you the implied return. However, this requires merging with your
# stock price data. We leave ptg_mean in the output so you can compute:
#
#   implied_return = (ptg_mean / current_price) - 1
#
# in your downstream pipeline where you have price data. Do NOT compute it
# here because we'd need to pull CRSP prices, handle ticker-PERMNO linking,
# and potentially introduce stale price issues.

# ── Drop intermediate columns, save ─────────────────────────────────────────
ptg_panel = ptg_panel.drop(columns=['ptg_std'], errors='ignore')
ptg_panel = ptg_panel.drop_duplicates(subset=['ticker', 'date'], keep='last')
ptg_panel.to_parquet('../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_price_targets.parquet', index=False, engine='pyarrow')

ptg_cols = [c for c in ptg_panel.columns if c not in ['ticker', 'date']]
print(f"\nSaved ../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_price_targets.parquet: {ptg_panel.shape}")
print(f"  Tickers: {ptg_panel['ticker'].nunique():,}")
print(f"  Date range: {ptg_panel['date'].min().date()} → {ptg_panel['date'].max().date()}")
print(f"  Factors ({len(ptg_cols)}):")
for c in ptg_cols:
    pct_nan = ptg_panel[c].isna().mean() * 100
    print(f"    {c:<30s} — {pct_nan:>5.1f}% NaN")


# ═══════════════════════════════════════════════════════════════════════════════
# PART 4: IBES TICKER → PERMNO LINKING TABLE
# ═══════════════════════════════════════════════════════════════════════════════
#
# Your output files use IBES tickers. To merge with CRSP or your equity universe,
# you need a linking table. This pulls the standard IBES-CRSP crosswalk.
#
# The link goes: IBES ticker → CUSIP (via ibes.idsum) → PERMNO (via crsp.stocknames).
# We save this once so you can reuse it for all three datasets.
#
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'=' * 80}")
print("PART 4: IBES → PERMNO LINKING TABLE")
print("=" * 80)

print("\nBuilding IBES ticker → CRSP PERMNO crosswalk...")
linktable = db.raw_sql("""
    SELECT a.ticker, a.cusip AS ibes_cusip, a.sdates,
           b.permno, b.ncusip
    FROM ibes.idsum a
    INNER JOIN crsp.stocknames b
        ON a.cusip = b.ncusip
    WHERE a.usfirm = 1
""")

# Convert start dates to datetime
linktable['sdates'] = pd.to_datetime(linktable['sdates'])

# Safely compute 'edates' using pandas by looking at the next identifier's start date
linktable = linktable.sort_values(['ticker', 'sdates'])
linktable['edates'] = linktable.groupby('ticker')['sdates'].shift(-1)
linktable['edates'] = linktable['edates'].fillna(pd.Timestamp('2099-12-31'))

linktable.to_parquet('../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_permno_link.parquet', index=False, engine='pyarrow')
print(f"  Saved ../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_permno_link.parquet: {linktable.shape}")
print(f"  Unique IBES tickers: {linktable['ticker'].nunique():,}")
print(f"  Unique PERMNOs: {linktable['permno'].nunique():,}")

# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print(f"\n{'=' * 80}")
print("COLLECTION COMPLETE")
print("=" * 80)
print(f"""
Files saved:
  data/ibes_revenue.parquet          — Revenue estimate factors
  data/ibes_recommendations.parquet  — Analyst recommendation factors
  data/ibes_price_targets.parquet    — Price target factors
  data/ibes_permno_link.parquet      — IBES ticker → CRSP PERMNO crosswalk

All datasets keyed on (ticker, date) where:
  • 'ticker' = IBES ticker identifier
  • 'date'   = month-end date (point-in-time safe)

To merge with CRSP/your equity universe:
  1. Load ibes_permno_link.parquet
  2. For each (ticker, date) in the factor file, find the matching PERMNO
     where link.sdates <= date <= link.edates
  3. Merge on PERMNO with your price/return data

FACTOR INVENTORY:

  Revenue ({len(rev_cols)} factors):
    rev_mean_fy1         — Consensus mean revenue estimate (current FY)
    rev_numest           — Number of analysts covering revenue
    rev_dispersion       — Analyst disagreement on revenue (stdev/|mean|)
    rev_range            — Revenue estimate range ((high-low)/|mean|)
    rev_revision_1m      — 1-month revision in revenue consensus (%)
    rev_revision_3m      — 3-month revision in revenue consensus (%)
    rev_numest_chg       — Change in # of analysts covering revenue
    rev_fy2_revision_1m  — Next-FY revenue revision (%)
    rev_fy2_dispersion   — Next-FY revenue dispersion
    rev_q_revision_1m    — Current-quarter revenue revision (%)
    rev_q_dispersion     — Current-quarter revenue dispersion
    rev_eps_divergence   — Revenue revision minus EPS revision (quality signal)
    rev_surprise         — Most recent revenue surprise (actual - consensus)
    rev_surprise_pct     — Revenue surprise as % of consensus

  Recommendations ({len(rec_cols)} factors):
    rec_mean             — Consensus recommendation (1=Strong Buy, 5=Sell)
    rec_median           — Median recommendation
    rec_numrec           — Number of active recommendations
    rec_buy_pct          — % of analysts with Buy/Strong Buy
    rec_sell_pct         — % of analysts with Sell/Underperform
    rec_buy_sell_spread  — Buy% minus Sell% (net bullishness)
    rec_dispersion       — Std dev of recommendation codes
    rec_revision         — 1-month change in consensus recommendation
    rec_revision_3m      — 3-month change in consensus recommendation
    rec_upgrades         — Count of upgrades this month
    rec_downgrades       — Count of downgrades this month
    rec_changes          — Total recommendation changes this month
    rec_up_down_ratio    — Upgrades / Downgrades ratio
    rec_breadth          — (Upgrades - Downgrades) / Total changes

  Price Targets ({len(ptg_cols)} factors):
    ptg_mean             — Consensus mean price target ($)
    ptg_median           — Median price target ($)
    ptg_high             — Highest analyst price target ($)
    ptg_low              — Lowest analyst price target ($)
    ptg_numest           — Number of active price targets
    ptg_dispersion       — Price target disagreement (std/mean)
    ptg_range            — Normalized range ((high-low)/mean)
    ptg_upside_skew      — Upside vs downside asymmetry
    ptg_revision         — 1-month consensus target revision (%)
    ptg_revision_3m      — 3-month consensus target revision (%)
    ptg_numest_chg       — Change in # of analysts with targets

NOTE ON IMPLIED RETURN:
  ptg_mean is included so you can compute the most powerful price target signal:
    implied_return = (ptg_mean / current_stock_price) - 1
  Do this in your pipeline where you have price data, not here.
""")

db.close()

Loading library list...
Done
PART 1: REVENUE ESTIMATES

Pulling revenue summary statistics...
  Revenue summary: 3,319,618 rows, 12,447 tickers
  FPI distribution:
fpi
1    1152955
2    1127636
6    1039027

Pulling revenue actuals...
  Revenue actuals: 669,852 rows

Pulling EPS summary (for revenue-EPS divergence)...
  EPS summary: 1,206,656 rows

Computing revenue factors...

Saved data/ibes_revenue.parquet: (1150491, 17)
  Tickers: 12,397
  Date range: 2003-01-16 → 2024-12-19
  Factors (15):
    rev_mean_fy1                   —   0.0% NaN
    numest                         —   0.0% NaN
    rev_dispersion                 —  19.2% NaN
    rev_range                      —   2.7% NaN
    rev_revision_1m                —   9.5% NaN
    rev_revision_3m                —  27.7% NaN
    rev_numest                     —   0.0% NaN
    rev_numest_chg                 —   9.5% NaN
    rev_fy2_revision_1m            —  11.9% NaN
    rev_fy2_dispersion             —  20.8% NaN
    rev_q_revision_1

In [ ]:
"""
Post-collection cleanup: fix structural NaNs, forward-fill surprises,
document OAP replacements.

Key insight from review:
  1. rec_up_down_ratio / rec_breadth NaNs are division-by-zero (0 changes
     in the month), NOT missing data. Fill with 0.
  2. rev_surprise 99% NaN is because companies only report ~4x/year.
     Forward-fill carries the most recent surprise forward, matching
     how OAP computes RevenueSurprise (12% NaN).
"""

import pandas as pd
import numpy as np

# ── Load ─────────────────────────────────────────────────────────────────────
rev = pd.read_parquet('data/ibes_revenue.parquet')
rec = pd.read_parquet('data/ibes_recommendations.parquet')
ptg = pd.read_parquet('data/ibes_price_targets.parquet')

# ═══════════════════════════════════════════════════════════════════════════════
# REVENUE: Forward-fill surprises within each ticker
# ═══════════════════════════════════════════════════════════════════════════════
# rev_surprise is only populated on the month the actual was announced (~4x/yr).
# Forward-fill so the "most recent surprise" carries forward — this is exactly
# what OAP does with RevenueSurprise, and is economically correct (the market
# continues to price in the most recent surprise until the next one arrives).

rev = rev.sort_values(['ticker', 'date'])
for col in ['rev_surprise', 'rev_surprise_pct']:
    # limit=4: carry surprise forward for ~4 months (one earnings cycle).
    # After that, if no new actual arrives, it reverts to NaN rather than
    # feeding stale data from a delisted/acquired company into the model.
    rev[col] = rev.groupby('ticker')[col].ffill(limit=4)

rev.to_parquet('../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_revenue.parquet', index=False, engine='pyarrow')

# ═══════════════════════════════════════════════════════════════════════════════
# RECOMMENDATIONS: Fill structural zeros
# ═══════════════════════════════════════════════════════════════════════════════
# rec_up_down_ratio = upgrades / downgrades → NaN when downgrades == 0
# rec_breadth = (upgrades - downgrades) / total_changes → NaN when changes == 0
#
# These are NOT missing data. A stock with 0 recommendation changes in a month
# has breadth of exactly 0 and a ratio that is undefined but economically neutral.
# Filling with 0 is correct: "no signal this month" = neutral.

# rec_breadth = (upgrades - downgrades) / total_changes → NaN when changes == 0
# This IS a structural zero: "no signal this month" = neutral. Fill with 0.
#
# rec_up_down_ratio = upgrades / downgrades → NaN in TWO different scenarios:
#   (a) 0 upgrades, 0 downgrades → genuinely neutral → should be 0
#   (b) 5 upgrades, 0 downgrades → extremely bullish → filling 0 would be WRONG
# Because we cannot distinguish (a) from (b) after the fact without going back
# to the raw counts, and rec_breadth already handles this correctly (bounded
# -1 to +1, no zero-denominator issue), we DROP rec_up_down_ratio entirely.

rec = rec.drop(columns=['rec_up_down_ratio'], errors='ignore')
rec['rec_breadth'] = rec['rec_breadth'].fillna(0)

rec.to_parquet('../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_recommendations.parquet', index=False, engine='pyarrow')

# ═══════════════════════════════════════════════════════════════════════════════
# PRINT FINAL INVENTORIES
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 80)
print("FINAL IBES FACTOR INVENTORY (after cleanup)")
print("=" * 80)

for name, df in [('Revenue', rev), ('Recommendations', rec), ('Price Targets', ptg)]:
    cols = [c for c in df.columns if c not in ['ticker', 'date']]
    print(f"\n{name} ({len(cols)} factors, {df.shape[0]:,} rows, "
          f"{df['ticker'].nunique():,} tickers):")
    for c in cols:
        pct = df[c].isna().mean() * 100
        print(f"  {c:<30s} {pct:>5.1f}% NaN")

# ── OAP factors to replace ──────────────────────────────────────────────────
print(f"\n{'=' * 80}")
print("OAP FACTORS TO REPLACE WITH IBES (dramatically better coverage)")
print("=" * 80)

replacements = {
    'ConsRecomm':              ('rec_mean',       '86.5% → 0.0%'),
    'ChangeInRecommendation':  ('rec_revision',   '39.6% → 1.2%'),
    'UpRecomm':                ('rec_upgrades',   '39.6% → 0.0%'),
    'DownRecomm':              ('rec_downgrades', '39.6% → 0.0%'),
}
for oap, (ibes, improvement) in replacements.items():
    print(f"  {oap:<30s} → {ibes:<20s} (NaN: {improvement})")

print(f"\n{'=' * 80}")
print("OAP FACTORS TO KEEP ALONGSIDE IBES (no overlap)")
print("=" * 80)
keep_oap = [
    ('RevenueSurprise',            '12.1%', 'Keep — or use IBES rev_surprise_pct after ffill'),
    ('FEPS',                        '4.0%', 'EPS forecast level'),
    ('ForecastDispersion',          '4.1%', 'EPS disagreement'),
    ('AnalystRevision',             '4.1%', 'EPS revision'),
    ('REV6',                        '4.9%', 'Earnings revisions'),
    ('EarningsSurprise',            '6.8%', 'EPS surprise'),
    ('fgr5yrLag',                  '17.9%', 'Long-term EPS growth'),
    ('AOP',                        '22.3%', 'Analyst optimism'),
    ('AnalystValue',               '22.3%', 'Analyst-implied value'),
    ('EarningsForecastDisparity',  '23.3%', 'Long vs short EPS'),
    ('PredictedFE',                '29.3%', 'Predicted forecast error'),
]
for name, nan_pct, desc in keep_oap:
    print(f"  {name:<30s} {nan_pct:>6s} NaN  — {desc}")

FINAL IBES FACTOR INVENTORY (after cleanup)

Revenue (15 factors, 1,150,491 rows, 12,397 tickers):
  rev_mean_fy1                     0.0% NaN
  numest                           0.0% NaN
  rev_dispersion                  19.2% NaN
  rev_range                        2.7% NaN
  rev_revision_1m                  9.5% NaN
  rev_revision_3m                 27.7% NaN
  rev_numest                       0.0% NaN
  rev_numest_chg                   9.5% NaN
  rev_fy2_revision_1m             11.9% NaN
  rev_fy2_dispersion              20.8% NaN
  rev_q_revision_1m               40.6% NaN
  rev_q_dispersion                28.1% NaN
  rev_eps_divergence              10.4% NaN
  rev_surprise                    97.5% NaN
  rev_surprise_pct                97.6% NaN

Recommendations (13 factors, 4,257,974 rows, 51,457 tickers):
  rec_mean                         0.0% NaN
  rec_median                       0.0% NaN
  rec_numrec                       0.0% NaN
  rec_buy_pct                      0.0% NaN
  